In [3]:
import pandas as pd 
import os
import platform
import sqlite3

In [4]:
if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'
    
TOKENIZER_ROOT = os.path.join(ROOT, "data")
OUT_ROOT = os.path.join(ROOT, "output_dump")
RESULTS_ROOT = os.path.join(ROOT, "results")

SQL_DB = os.path.join(RESULTS_ROOT, "results.db")


def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)

In [5]:
words_df = pd.read_sql_query("SELECT * FROM WordDetails", conn)
words_df

,WordUID,Word,CharacterLength,LogFrequencies,QuintilesA1,QuintilesA2
0,1,if,2,5.256744,4,4
1,2,you,3,6.329340,4,4
2,3,were,4,4.928421,4,4
3,4,to,2,6.063172,4,4
4,5,journey,7,3.007748,3,3
...,...,...,...,...,...,...
10134,10135,disks,5,1.913814,2,1
10135,10136,stored,6,2.230449,2,2
10136,10137,embrace,7,2.587711,3,2
10137,10138,coincides,9,1.146128,1,0


In [ ]:
#Create Quintiles (q=5) based on LogFrequencies column

#Note: Issues with this approach is that the log frequencies are derived from (<Fill in>) and so has a many words that are not there in the distribution, which are then assigned 0 by default. For example numbers like 8, 1000 etc. So, all of these get assigned to the lowest quintile. 

# Alternate approach is to tag values that have 0 Logfrequencies and skip those to assign quintiles and use only the non zero values to create quintiles and also use in subsequent analysis.

#Approach 1
words_df['LogFrequencies'] = words_df['LogFrequencies'].astype(float)
words_df['quintiles_a1'] = pd.qcut(words_df['LogFrequencies'], 5, labels=False)
words_df['quintiles_a1'] = words_df['quintiles_a1'].astype(int)
#Approach 2

words_df = words_df.merge(words_df[words_df['LogFrequencies'] > 0][["WordUID", "LogFrequencies"]].copy().assign(quintiles_a2 = lambda x: pd.qcut(x['LogFrequencies'], 5, labels=False)).drop(columns=['LogFrequencies']), on='WordUID', how='left')

words_df['quintiles_a2'] = words_df['quintiles_a2'].fillna(-1).astype(int)


words_df

In [ ]:
#Column datatypes
words_df.dtypes

In [ ]:
# # #Write back quintiles_a1 and quintiles_a2 to the database

# for row in words_df[['WordUID', 'quintiles_a1', 'quintiles_a2']].to_dict(orient='records'):
#     c.execute("UPDATE WordDetails SET QuintilesA1 = ?, QuintilesA2 = ? WHERE WordUID = ?", (row['quintiles_a1'], row['quintiles_a2'], row['WordUID']))
# conn.commit()



In [ ]:
#Alternative Approach - 
# Use spacy's en_core_web_lg model to calculate get token.probs and use that to calculate the quintiles.

words_df

In [1]:
import spacy
from spacy.lookups import load_lookups

nlp = spacy.load("en_core_web_sm")


# Load the probability table from spacy-lookups-data
lookups = load_lookups("en", tables=["lexeme_prob"])
# Add the probability table to the model’s vocab
nlp.vocab.lookups.add_table("lexeme_prob", lookups.get_table("lexeme_prob"))


Table([(12646065887601541794, -3.0678977966),
       (2593208677638477497, -3.454959631),
       (7425985699627899538, -3.5287666321),
       (4690420944186131903, -3.7915651798),
       (3791531372978436496, -3.8560216427),
       (11901859001352538922, -3.9297883511),
       (2283656566040971221, -4.1131081581),
       (886050111519832510, -4.275873661),
       (7624161793554793053, -4.3737912178),
       (10239237003504588839, -4.3880500793),
       (3411606890003347522, -4.4577488899),
       (4380130941430378203, -4.4645047188),
       (908432558851201422, -4.6065607071),
       (3002984154512732771, -4.6190719604),
       (16428057658620181782, -4.8305592537),
       (2043519015752540944, -4.8599386214999996),
       (16037325823156266367, -4.8801093102),
       (15884554869126768810, -5.0267758369),
       (8205403955989537350, -5.0592465401),
       (8532415787641010193, -5.1291651726),
       (14692702688101715474, -5.1564846039),
       (5640369432778651323, -5.1727361679),
 

In [10]:
from tqdm.auto import tqdm

spacy_log_probs = []
for word in tqdm(words_df.iterrows()):
    token = nlp(word[1]['Word'])
    if len(token) == 1:
        spacy_log_probs.append({"word_uid": word[1]['WordUID'], "word": word[1]['Word'], "log_prob": token[0].prob, "token_count": len(token)})
    elif len(token) > 1:
        # If the tokenization results in multiple tokens, calculate the average log probability
        sum_log_prob = sum([t.prob for t in token])
        spacy_log_probs.append({"word_uid": word[1]['WordUID'], "word": word[1]['Word'], "log_prob": sum_log_prob, "token_count": len(token)})
    else:
        spacy_log_probs.append({"word_uid": word[1]['WordUID'], "word": word[1]['Word'], "log_prob": None, "token_count": 0})
spacy_log_probs_df = pd.DataFrame(spacy_log_probs)

0it [00:00, ?it/s]

In [29]:
[(k.text, k.prob) for k in nlp("england")]

[('england', -13.3741846085)]

In [12]:
spacy_log_probs_df

,word_uid,word,log_prob,token_count
0,1,if,-5.763590,1
1,2,you,-4.373791,1
2,3,were,-6.673175,1
3,4,to,-3.856022,1
4,5,journey,-11.295086,1
...,...,...,...,...
10134,10135,disks,-12.883524,1
10135,10136,stored,-11.588490,1
10136,10137,embrace,-11.984515,1
10137,10138,coincides,-14.459026,1


In [16]:
#use  log_prob to calculate quintiles
spacy_log_probs_df['log_prob'] = spacy_log_probs_df['log_prob'].astype(float)
spacy_log_probs_df['quintiles_a3'] = pd.qcut(spacy_log_probs_df['log_prob'], 5, labels=False)
spacy_log_probs_df['quintiles_a3'] = spacy_log_probs_df['quintiles_a3'].astype(int)

#Join on words_df on word_uid
spacy_log_probs_df = spacy_log_probs_df.merge(words_df, left_on='word_uid', right_on='WordUID', how='left')
spacy_log_probs_df

,word_uid,word,log_prob,token_count,quintiles_a3,WordUID,Word,CharacterLength,LogFrequencies,QuintilesA1,QuintilesA2
0,1,if,-5.763590,1,4,1,if,2,5.256744,4,4
1,2,you,-4.373791,1,4,2,you,3,6.329340,4,4
2,3,were,-6.673175,1,4,3,were,4,4.928421,4,4
3,4,to,-3.856022,1,4,4,to,2,6.063172,4,4
4,5,journey,-11.295086,1,3,5,journey,7,3.007748,3,3
...,...,...,...,...,...,...,...,...,...,...,...
10134,10135,disks,-12.883524,1,2,10135,disks,5,1.913814,2,1
10135,10136,stored,-11.588490,1,3,10136,stored,6,2.230449,2,2
10136,10137,embrace,-11.984515,1,2,10137,embrace,7,2.587711,3,2
10137,10138,coincides,-14.459026,1,1,10138,coincides,9,1.146128,1,0


In [19]:
spacy_log_probs_df["A1_A2_mismatch"] = spacy_log_probs_df.apply(lambda x: 1 if x['QuintilesA1'] != x['QuintilesA2'] else 0, axis=1)
spacy_log_probs_df["A1_A3_mismatch"] = spacy_log_probs_df.apply(lambda x: 1 if x['QuintilesA1'] != x['quintiles_a3'] else 0, axis=1)
spacy_log_probs_df["A2_A3_mismatch"] = spacy_log_probs_df.apply(lambda x: 1 if x['QuintilesA2'] != x['quintiles_a3'] else 0, axis=1)
spacy_log_probs_df["A1_A2_A3_mismatch"] = spacy_log_probs_df.apply(lambda x: 1 if (x['QuintilesA1'] != x['QuintilesA2']) and (x['QuintilesA1'] != x['quintiles_a3']) and (x['QuintilesA2'] != x['quintiles_a3']) else 0, axis=1)


spacy_log_probs_df

,word_uid,word,log_prob,token_count,quintiles_a3,WordUID,Word,CharacterLength,LogFrequencies,QuintilesA1,QuintilesA2,A1_A2_mismatch,A1_A3_mismatch,A2_A3_mismatch,A1_A2_A3_mismatch
0,1,if,-5.763590,1,4,1,if,2,5.256744,4,4,0,0,0,0
1,2,you,-4.373791,1,4,2,you,3,6.329340,4,4,0,0,0,0
2,3,were,-6.673175,1,4,3,were,4,4.928421,4,4,0,0,0,0
3,4,to,-3.856022,1,4,4,to,2,6.063172,4,4,0,0,0,0
4,5,journey,-11.295086,1,3,5,journey,7,3.007748,3,3,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10134,10135,disks,-12.883524,1,2,10135,disks,5,1.913814,2,1,1,0,1,0
10135,10136,stored,-11.588490,1,3,10136,stored,6,2.230449,2,2,0,1,1,0
10136,10137,embrace,-11.984515,1,2,10137,embrace,7,2.587711,3,2,1,1,0,0
10137,10138,coincides,-14.459026,1,1,10138,coincides,9,1.146128,1,0,1,0,1,0


In [20]:
subltex_df = pd.read_excel("/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/reading_time_analysis/SUBTLEXusExcel2007.xlsx")
subltex_df

,Word,FREQcount,CDcount,FREQlow,Cdlow,SUBTLWF,Lg10WF,SUBTLCD,Lg10CD
0,the,1501908,8388,1339811,8388,29449.176471,6.176644,100.000000,3.923710
1,to,1156570,8383,1138435,8380,22677.843137,6.063172,99.940391,3.923451
2,a,1041179,8382,976941,8380,20415.274510,6.017526,99.928469,3.923399
3,you,2134713,8381,1595028,8376,41857.117647,6.329340,99.916547,3.923348
4,and,682780,8379,515365,8374,13387.843137,5.834281,99.892704,3.923244
...,...,...,...,...,...,...,...,...,...
74281,Zoroastrian,1,1,0,0,0.019608,0.301030,0.011922,0.301030
74282,Zoroastrianism,1,1,0,0,0.019608,0.301030,0.011922,0.301030
74283,zugzwang,1,1,1,1,0.019608,0.301030,0.011922,0.301030
74284,zygotes,1,1,1,1,0.019608,0.301030,0.011922,0.301030


In [21]:
spacy_log_probs_df["A3_ranks"] = spacy_log_probs_df["log_prob"].rank(method='first', ascending=False)
spacy_log_probs_df["A1_ranks"] = spacy_log_probs_df["LogFrequencies"].rank(method='first', ascending=False)

spacy_log_probs_df

,word_uid,word,log_prob,token_count,quintiles_a3,WordUID,Word,CharacterLength,LogFrequencies,QuintilesA1,QuintilesA2,A1_A2_mismatch,A1_A3_mismatch,A2_A3_mismatch,A1_A2_A3_mismatch,A3_ranks,A1_ranks
0,1,if,-5.763590,1,4,1,if,2,5.256744,4,4,0,0,0,0,30.0,48.0
1,2,you,-4.373791,1,4,2,you,3,6.329340,4,4,0,0,0,0,6.0,1.0
2,3,were,-6.673175,1,4,3,were,4,4.928421,4,4,0,0,0,0,77.0,90.0
3,4,to,-3.856022,1,4,4,to,2,6.063172,4,4,0,0,0,0,2.0,4.0
4,5,journey,-11.295086,1,3,5,journey,7,3.007748,3,3,0,0,0,0,3011.0,2160.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10134,10135,disks,-12.883524,1,2,10135,disks,5,1.913814,2,1,1,0,1,0,5485.0,5855.0
10135,10136,stored,-11.588490,1,3,10136,stored,6,2.230449,2,2,0,1,1,0,3492.0,4776.0
10136,10137,embrace,-11.984515,1,2,10137,embrace,7,2.587711,3,2,1,1,0,0,4142.0,3535.0
10137,10138,coincides,-14.459026,1,1,10138,coincides,9,1.146128,1,0,1,0,1,0,7307.0,7707.0


In [26]:
for row in spacy_log_probs_df[['WordUID', 'quintiles_a3', 'log_prob']].to_dict(orient='records'):
    c.execute("UPDATE WordDetails SET QuintilesA3 = ?, SpacyTokenProb = ? WHERE WordUID = ?", (row['quintiles_a3'], row['log_prob'], row['WordUID']))
conn.commit()